<a href="https://colab.research.google.com/github/AMLU-ANNA-JOSHY/Anomaly_Detection/blob/main/Elliptic_Bitcoin_fraud_detection_GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fraud Detection using GNN on Elliptic Bitcoin Dataset**



### **Elliptic Bitcoin Dataset**

- It is a labeled, graph-structured cryptocurrency fraud dataset built from the Bitcoin blockchain.
- The dataset captures a time-ordered graph of Bitcoin transactions.
- The Bitcoin transactions are represented  as nodes in a directed graph, and the directed flow of Bitcoin between transactions is represented via edges, and the 49 time steps present model the temporal evolution.
- Including time step(1-49) there are 166 features: 94 local and 74 aggregated features (non-local features derived from the transaction’s one-hop neighborhood)
- The available labels are licit, illicit, or unknown.
- Used for applications such as anomaly detection, illicit transaction prediction,  and graph-based fraud detection.

Ref: https://medium.com/elliptic/the-elliptic-data-set-opening-up-machine-learning-on-the-blockchain-e0a343d99a14

In [ ]:
!pip install kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d ellipticco/elliptic-data-set
!unzip -o elliptic-data-set.zip

In [2]:
import pandas as pd

features_df = pd.read_csv("elliptic_bitcoin_dataset/elliptic_txs_features.csv", header=None)
edges_df = pd.read_csv("elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv")
classes_df = pd.read_csv("elliptic_bitcoin_dataset/elliptic_txs_classes.csv")

print("features shape:", features_df.shape)
print("edges shape:", edges_df.shape)
print("classes shape:", classes_df.shape)

features shape: (203769, 167)
edges shape: (234355, 2)
classes shape: (203769, 2)


In [ ]:
# inspect features df

print("Features head:")
print(features_df.head())

Features head:
         0    1         2         3         4          5         6    \
0  230425980    1 -0.171469 -0.184668 -1.201369  -0.121970 -0.043875   
1    5530458    1 -0.171484 -0.184668 -1.201369  -0.121970 -0.043875   
2  232022460    1 -0.172107 -0.184668 -1.201369  -0.121970 -0.043875   
3  232438397    1  0.163054  1.963790 -0.646376  12.409294 -0.063725   
4  230460314    1  1.011523 -0.081127 -1.201369   1.153668  0.333276   

        7          8         9    ...       157       158       159       160  \
0 -0.113002  -0.061584 -0.162097  ... -0.562153 -0.600999  1.461330  1.461369   
1 -0.113002  -0.061584 -0.162112  ...  0.947382  0.673103 -0.979074 -0.978556   
2 -0.113002  -0.061584 -0.162749  ...  0.670883  0.439728 -0.979074 -0.978556   
3  9.782742  12.414558 -0.163645  ... -0.577099 -0.613614  0.241128  0.241406   
4  1.312656  -0.061584 -0.163523  ... -0.511871 -0.400422  0.517257  0.579382   

        161       162       163       164       165       166  
0

There are no column headers available, so add them explicitly. The first column is transaction id, second is time steps and the other 165 represent anonymous features.

In [3]:
cols = ["txId", "time_step"] + [f"f{i}" for i in range(165)]
features_df.columns = cols
print(features_df.head())

        txId  time_step        f0        f1        f2         f3        f4  \
0  230425980          1 -0.171469 -0.184668 -1.201369  -0.121970 -0.043875   
1    5530458          1 -0.171484 -0.184668 -1.201369  -0.121970 -0.043875   
2  232022460          1 -0.172107 -0.184668 -1.201369  -0.121970 -0.043875   
3  232438397          1  0.163054  1.963790 -0.646376  12.409294 -0.063725   
4  230460314          1  1.011523 -0.081127 -1.201369   1.153668  0.333276   

         f5         f6        f7  ...      f155      f156      f157      f158  \
0 -0.113002  -0.061584 -0.162097  ... -0.562153 -0.600999  1.461330  1.461369   
1 -0.113002  -0.061584 -0.162112  ...  0.947382  0.673103 -0.979074 -0.978556   
2 -0.113002  -0.061584 -0.162749  ...  0.670883  0.439728 -0.979074 -0.978556   
3  9.782742  12.414558 -0.163645  ... -0.577099 -0.613614  0.241128  0.241406   
4  1.312656  -0.061584 -0.163523  ... -0.511871 -0.400422  0.517257  0.579382   

       f159      f160      f161      f162   

In [ ]:
# inspect classes df

print("Classes head:")
print(classes_df.head())

print("\nUnique classes:")
print(classes_df["class"].unique())

Classes head:
        txId    class
0  230425980  unknown
1    5530458  unknown
2  232022460  unknown
3  232438397        2
4  230460314  unknown

Unique classes:
['unknown' '2' '1']


- The 3 classes are ['unknown', '2', '1'] => [unknown, licit, illicit(fraud)]

In [4]:
# Merge features with labels

df = features_df.merge(classes_df, on="txId")

In [5]:
# remove unknown rows

df = df[df["class"] != "unknown"]

In [6]:
# convert to binary labels: fraud = 1, licit = 0

df["class"] = df["class"].apply(lambda x: 1 if x == "1" else 0)


In [ ]:
# inspect again, previously (203769, 167)

print(df.shape)
print(df["class"].value_counts())

(46564, 168)
class
0    42019
1     4545
Name: count, dtype: int64


In [ ]:
# inspect edges df

print("Edges head:")
print(edges_df.head())
print("Number of entries:", len(edges_df))

Edges head:
       txId1      txId2
0  230425980    5530458
1  232022460  232438397
2  230460314  230459870
3  230333930  230595899
4  232013274  232029206
Number of entries: 234355


In [7]:
# remove enteries with txId1/2 removed in previous step

valid_nodes = set(df["txId"])

edges_df = edges_df[
    edges_df["txId1"].isin(valid_nodes) &
    edges_df["txId2"].isin(valid_nodes)
]

In [ ]:
# inspect edges df again, previous len: 234355

print("Edges head:")
print(edges_df.head())
print("Number of entries:", len(edges_df))

Edges head:
        txId1      txId2
5   232344069   27553029
8     3881097  232457116
15  232051089  232470704
26  230473487    7089694
33  231182296   14660781
Number of entries: 36624


### **GNN using PyTorch Geometric**

- Graph tensors needed for GNN are:
node feature matrix (x), edge index (edge_index), and label vector (y)

In [ ]:
!pip install torch-geometric

In [9]:
# create a mapping of transaction ids to integers

txid_to_idx = {txid: i for i, txid in enumerate(df["txId"].values)}

In [10]:
# node feature matrix: all features except txId and class label

import torch

feature_cols = [c for c in df.columns if c not in ["txId", "class"]]

x = torch.tensor(df[feature_cols].values, dtype=torch.float)
print(x.shape)

torch.Size([46564, 166])


In [11]:
# label vector

y = torch.tensor(df["class"].values, dtype=torch.long)
print(y.shape)

torch.Size([46564])


In [12]:
# edges to index (src-dest) pairs

src = edges_df["txId1"].map(txid_to_idx).values
dst = edges_df["txId2"].map(txid_to_idx).values

In [13]:
# edge_index of shape [2, no. of edges]

import numpy as np

edge_index = torch.tensor(np.array([src, dst]),dtype=torch.long)

print(edge_index.shape)

torch.Size([2, 36624])


In [14]:
# PyG data object

from torch_geometric.data import Data

data = Data(x=x, edge_index=edge_index, y=y)
print(data)

Data(x=[46564, 166], edge_index=[2, 36624], y=[46564])


In [ ]:
# check if mapping is correct

print("Num nodes:", data.num_nodes) # x.shape[0]
print("Num edges:", data.num_edges) # edge_index.shape[1]
print("Num node features:", data.num_node_features) # x.shape[1]
print("Unique labels:", torch.unique(data.y))

Num nodes: 46564
Num edges: 36624
Num node features: 166
Unique labels: tensor([0, 1])


In [15]:
# create train/test masks for data

import numpy as np
from sklearn.model_selection import train_test_split

node_indices = np.arange(data.num_nodes)

train_idx, test_idx = train_test_split(
    node_indices,
    test_size=0.2,
    stratify=data.y.numpy(),
    random_state=42
)

train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

In [16]:
# attach masks to graph and verify that 80-20 split appears
# and ensure fraud count was stratified

data.train_mask = train_mask
data.test_mask = test_mask

print("Train nodes:", int(data.train_mask.sum()))
print("Test nodes:", int(data.test_mask.sum()))

print("Train fraud count:", int(data.y[data.train_mask].sum()))
print("Test fraud count:", int(data.y[data.test_mask].sum()))

Train nodes: 37251
Test nodes: 9313
Train fraud count: 3636
Test fraud count: 909


## **Building GNNs**

- **Build GNN:** Each transaction is represented as a node vector with 166 features; each bitcoin flow is represented via edges; each transaction(node) has an associated label(fraud/licit)
- **GNN problem:** Binary node classification problem: Classify each node as fraud/licit.

In [20]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

### **Graph Convolutional Network (GCN)**

In [ ]:
from torch_geometric.nn import GCNConv

In [ ]:
# GCN model definition

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 2)  # 2 classes: normal, fraud

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)

        x = self.conv2(x, edge_index)
        return x

In [ ]:
# model initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)

model = GCN(in_channels=data.num_node_features, hidden_channels=64).to(device)

In [ ]:
# assign class-weights since there is class imbalance
# give more weightage to the minority fraud class

num_normal = int((data.y[data.train_mask] == 0).sum())
num_fraud = int((data.y[data.train_mask] == 1).sum())

class_weights = torch.tensor(
    [1.0, num_normal / num_fraud],
    dtype=torch.float
).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [ ]:
# training function

def train():
    model.train()
    optimizer.zero_grad()

    out = model(data)   # shape: [num_nodes, 2]
    loss = criterion(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
# evaluation function

@torch.no_grad()
def evaluate(mask):
    model.eval()
    out = model(data)

    preds = out[mask].argmax(dim=1)
    probs = F.softmax(out[mask], dim=1)[:, 1]

    y_true = data.y[mask].cpu().numpy()
    y_pred = preds.cpu().numpy()
    y_prob = probs.cpu().numpy()

    return y_true, y_pred, y_prob

In [ ]:
# training

for epoch in range(1, 51):
    loss = train()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 10, Loss: 0.5951
Epoch 20, Loss: 0.3810
Epoch 30, Loss: 0.3169
Epoch 40, Loss: 0.2872
Epoch 50, Loss: 0.2742


In [ ]:
# testing

y_true, y_pred, y_prob = evaluate(data.test_mask)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_true, y_prob))

[[7411  993]
 [  92  817]]
              precision    recall  f1-score   support

           0     0.9877    0.8818    0.9318      8404
           1     0.4514    0.8988    0.6010       909

    accuracy                         0.8835      9313
   macro avg     0.7196    0.8903    0.7664      9313
weighted avg     0.9354    0.8835    0.8995      9313

ROC-AUC: 0.9574765068129849


In [ ]:
# threshold tuning to get best recall to avoid false positives

import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
best_recall = 0
best_thresh = 0

for t in thresholds:
    y_pred_thr = (y_prob >= t).astype(int)
    r = recall_score(y_true, y_pred_thr)
    if best_recall < r:
        best_recall = r
        best_thresh = t

print("Best threshold: ", best_thresh)

y_pred_thr = (y_prob >= best_thresh).astype(int)

p = precision_score(y_true, y_pred_thr)
r = recall_score(y_true, y_pred_thr)
f1 = f1_score(y_true, y_pred_thr)
cm = confusion_matrix(y_true, y_pred_thr)

print("\nPrecision:", round(p, 4))
print("Recall   :", round(r, 4))
print("F1       :", round(f1, 4))
print("\nConfusion matrix:")
print(cm)

Best threshold:  0.3

Precision: 0.3167
Recall   : 0.9472
F1       : 0.4746

Confusion matrix:
[[6546 1858]
 [  48  861]]


### **GraphSAGE**

In [18]:
from torch_geometric.nn import SAGEConv

In [29]:
# GraphSAGE model definition

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, 2)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)

        x = self.conv2(x, edge_index)
        return x

In [30]:
# model initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)

model = GraphSAGE(in_channels=data.num_node_features, hidden_channels=64).to(device)

In [31]:
# assign class-weights since there is class imbalance
# give more weightage to the minority fraud class

num_normal = int((data.y[data.train_mask] == 0).sum())
num_fraud = int((data.y[data.train_mask] == 1).sum())

class_weights = torch.tensor(
    [1.0, num_normal / num_fraud],
    dtype=torch.float
).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [32]:
# training function

def train():
    model.train()
    optimizer.zero_grad()

    out = model(data)   # shape: [num_nodes, 2]
    loss = criterion(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()
    return loss.item()

In [33]:
# evaluation function

@torch.no_grad()
def evaluate(mask):
    model.eval()
    out = model(data)

    preds = out[mask].argmax(dim=1)
    probs = F.softmax(out[mask], dim=1)[:, 1]

    y_true = data.y[mask].cpu().numpy()
    y_pred = preds.cpu().numpy()
    y_prob = probs.cpu().numpy()

    return y_true, y_pred, y_prob

In [34]:
# training

for epoch in range(1, 51):
    loss = train()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 10, Loss: 0.3344
Epoch 20, Loss: 0.2567
Epoch 30, Loss: 0.2136
Epoch 40, Loss: 0.1824
Epoch 50, Loss: 0.1615


In [35]:
# testing

y_true, y_pred, y_prob = evaluate(data.test_mask)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_true, y_prob))

[[7916  488]
 [  92  817]]
              precision    recall  f1-score   support

           0     0.9885    0.9419    0.9647      8404
           1     0.6261    0.8988    0.7380       909

    accuracy                         0.9377      9313
   macro avg     0.8073    0.9204    0.8513      9313
weighted avg     0.9531    0.9377    0.9425      9313

ROC-AUC: 0.9779848403688537


In [36]:
# threshold tuning to get best recall to avoid false positives

import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
best_recall = 0
best_thresh = 0

for t in thresholds:
    y_pred_thr = (y_prob >= t).astype(int)
    r = recall_score(y_true, y_pred_thr)
    if best_recall < r:
        best_recall = r
        best_thresh = t

print("Best threshold: ", best_thresh)

y_pred_thr = (y_prob >= best_thresh).astype(int)

p = precision_score(y_true, y_pred_thr)
r = recall_score(y_true, y_pred_thr)
f1 = f1_score(y_true, y_pred_thr)
cm = confusion_matrix(y_true, y_pred_thr)

print("\nPrecision:", round(p, 4))
print("Recall   :", round(r, 4))
print("F1       :", round(f1, 4))
print("\nConfusion matrix:")
print(cm)

Best threshold:  0.3

Precision: 0.4847
Recall   : 0.9406
F1       : 0.6397

Confusion matrix:
[[7495  909]
 [  54  855]]
